# 02 — Build Analytics Tools

This notebook validates the deterministic analytics engine for the Medical Insight Explorer Agent.

The purpose is to compute real healthcare metrics from cleaned relational Parquet tables before adding any LLM layer.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from agent.data_loader import HealthcareDataLoader
from agent.analytics_engine import HealthcareAnalyticsEngine

In [2]:
loader = HealthcareDataLoader(data_dir=PROJECT_ROOT / "data" / "processed")
tables = loader.load_tables()

engine = HealthcareAnalyticsEngine(tables)

In [3]:
engine.get_table_shapes()

,table_name,rows,columns
0,test_beneficiary,63968,26
1,test_inpatient,9551,30
2,test_labels,1353,1
3,test_outpatient,125841,27
4,train_beneficiary,138556,26
5,train_inpatient,40474,30
6,train_labels,5410,2
7,train_outpatient,517737,27


In [4]:
engine.inpatient_claim_summary()

{'total_claims': 40474,
 'unique_beneficiaries': 31289,
 'unique_claims': 40474,
 'unique_providers': 2092,
 'mean_reimbursement': 8960.641893561298,
 'median_reimbursement': 7000.0,
 'total_reimbursement': 362673020.0}

In [5]:
engine.outpatient_claim_summary()

{'total_claims': 517737,
 'unique_beneficiaries': 133980,
 'unique_claims': 517737,
 'unique_providers': 5012,
 'mean_reimbursement': 147.598240033067,
 'median_reimbursement': 80.0,
 'total_reimbursement': 76417070.0}

In [6]:
engine.beneficiary_age_summary()

{'count': 138556,
 'mean_age': 83.10368370911401,
 'median_age': 84.0,
 'min_age': 36.0,
 'max_age': 111.0}

In [7]:
engine.top_providers_by_claim_count(claim_type="inpatient", top_n=10)

,Provider,claim_count
0,PRV52019,516
1,PRV55462,386
2,PRV54367,322
3,PRV53706,282
4,PRV55209,275
5,PRV56560,248
6,PRV54742,231
7,PRV55230,225
8,PRV52340,224
9,PRV51501,223


In [8]:
engine.top_providers_by_claim_count(claim_type="outpatient", top_n=10)

,Provider,claim_count
0,PRV51459,8240
1,PRV53797,4739
2,PRV51574,4444
3,PRV53918,3588
4,PRV54895,3433
5,PRV55215,3250
6,PRV56011,2833
7,PRV52064,2806
8,PRV55004,2396
9,PRV57306,2315


In [9]:
engine.available_beneficiary_columns()

['BeneID',
 'DOB',
 'DOD',
 'Gender',
 'Race',
 'RenalDiseaseIndicator',
 'State',
 'County',
 'NoOfMonths_PartACov',
 'NoOfMonths_PartBCov',
 'ChronicCond_Alzheimer',
 'ChronicCond_Heartfailure',
 'ChronicCond_KidneyDisease',
 'ChronicCond_Cancer',
 'ChronicCond_ObstrPulmonary',
 'ChronicCond_Depression',
 'ChronicCond_Diabetes',
 'ChronicCond_IschemicHeart',
 'ChronicCond_Osteoporasis',
 'ChronicCond_rheumatoidarthritis',
 'ChronicCond_stroke',
 'IPAnnualReimbursementAmt',
 'IPAnnualDeductibleAmt',
 'OPAnnualReimbursementAmt',
 'OPAnnualDeductibleAmt',
 'AgeAtDeathOrLastClaim']

In [10]:
engine.average_inpatient_cost_by_chronic_condition("ChronicCond_Diabetes")

,ChronicCond_Diabetes,count,mean,median,sum
1,2,8012,9002.387668,7000.0,72127130
0,1,32462,8950.338550,7000.0,290545890


In [11]:
engine.claim_distribution_by_state(claim_type="inpatient").head(10)

,State,claim_count
4,5,3468
9,10,3056
43,45,2689
32,33,2597
38,39,1839
35,36,1699
13,14,1693
33,34,1442
10,11,1233
22,23,1134


## Result

The analytics engine successfully computes deterministic healthcare metrics from the cleaned relational Parquet tables.

These functions will later be exposed as controlled tools for the LLM layer, reducing hallucination risk by forcing numerical answers to come from pandas computations.